In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from tensorflow import keras

model = keras.models.load_model("mobilenetv2_se_besttuned.keras")  # change name
model.summary()

In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(model, show_shapes=True, show_layer_names=True)

In [ ]:
!pip install pydot
!apt-get install graphviz

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import load_model

In [ ]:
def h_sigmoid(x):
    return tf.nn.relu6(x + 3.0) / 6.0

def h_swish(x):
    return x * h_sigmoid(x)

@tf.keras.utils.register_keras_serializable(package="Custom")
class CoordAttFast(layers.Layer):

    def __init__(self, reduction=64, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        c = int(input_shape[-1])
        mip = max(8, c // self.reduction)

        self.conv1 = layers.Conv2D(mip, 1, padding="same", use_bias=True)
        self.conv_h = layers.Conv2D(c, 1, padding="same", use_bias=True)
        self.conv_w = layers.Conv2D(c, 1, padding="same", use_bias=True)

        super().build(input_shape)

    def call(self, x):

        h = tf.shape(x)[1]
        w = tf.shape(x)[2]

        x_h = tf.reduce_mean(x, axis=2, keepdims=True)
        x_w = tf.reduce_mean(x, axis=1, keepdims=True)
        x_w = tf.transpose(x_w, [0,2,1,3])

        y = tf.concat([x_h, x_w], axis=1)
        y = h_swish(self.conv1(y))

        y_h, y_w = tf.split(y, [h, w], axis=1)
        y_w = tf.transpose(y_w, [0,2,1,3])

        a_h = tf.sigmoid(self.conv_h(y_h))
        a_w = tf.sigmoid(self.conv_w(y_w))

        return x * a_h * a_w

    def get_config(self):
        config = super().get_config()
        config.update({"reduction": self.reduction})
        return config

In [ ]:
# CA
model_ca = load_model(
    "mobilenetv2_ca_tuned.keras",
    custom_objects={"CoordAttFast": CoordAttFast},
    compile=False
)
model.summary()

In [ ]:
plot_model(model_ca, show_shapes=True, show_layer_names=True)

In [ ]:
#ECA
from google.colab import files
uploaded = files.upload()

In [ ]:
model_eca = keras.models.load_model("mobilenetv2_eca_besttuned.keras")  # change name
model_eca.summary()

In [ ]:
plot_model(model_eca, show_shapes=True, show_layer_names=True)

In [ ]:
#Baseline
from google.colab import files
uploaded = files.upload()

In [ ]:
model_b = keras.models.load_model("mobilenetv2_tuned_best.keras")  # change name
model_b.summary()

In [ ]:
plot_model(model_b, show_shapes=True, show_layer_names=True)

In [ ]:
for layer in model.layers:
    print(layer.name)

In [ ]:
for layer in model_b.layers:
    print(layer.name)

In [ ]:
for layer in model_ca.layers:
    print(layer.name)

In [ ]:
for layer in model_eca.layers:
    print(layer.name)

In [ ]:
#gradcam++ code

In [ ]:

from google.colab import files
import zipfile, os

uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/images')

image_dir = '/content/images'
for root, dirs, files_list in os.walk('/content/images'):
    if any(f.lower().endswith(('.jpg','.jpeg','.png')) for f in files_list):
        image_dir = root
        break

image_files = sorted([f for f in os.listdir(image_dir)
                       if f.lower().endswith(('.jpg','.jpeg','.png'))])
print("Image dir :", image_dir)
print("Images    :", image_files)

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("mixed_float16")
print("TF version:", tf.__version__)

In [ ]:


models_dict = {
    "BASELINE_MODEL": model_b,
    "SE_MODEL"      : model,
    "ECA_MODEL"     : model_eca,
    "CA_MODEL"      : model_ca
}
print("Models loaded:", list(models_dict.keys()))

In [ ]:

IMG_SIZE    = (224, 224)
CLASS_NAMES = {0:"akiec", 1:"bcc", 2:"bkl", 3:"df", 4:"mel", 5:"nv", 6:"vasc"}


TARGET_LAYERS = {
    "BASELINE_MODEL" : "global_average_pooling2d_50",  
    "SE_MODEL"       : "multiply",
    "ECA_MODEL"      : "multiply",
    "CA_MODEL"       : "coord_att_fast",
}

In [ ]:

print("--- Layer name verification ---")
for name, mdl in models_dict.items():
    target = TARGET_LAYERS[name]
    found  = any(l.name == target for l in mdl.layers)
    status = "" if found else " NOT FOUND"
    print(f"  {name:20s}  target='{target}'  {status}")
    if not found:
        print(f"    Available layers: {[l.name for l in mdl.layers]}")

In [ ]:

mobilenet_preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

def preprocess_image(img_path, target_size=IMG_SIZE):
    
    img         = load_img(img_path, target_size=target_size)
    img_array   = img_to_array(img)                       
    img_display = img_array / 255.0                       
    img_input   = mobilenet_preprocess(img_array.copy())  
    return np.expand_dims(img_input, axis=0), img_display

In [ ]:

def build_grad_model(model, target_layer_name):
    """
    SE / ECA / CA : single joint model  [features, predictions]
    BASELINE      : two models — feature_extractor + classifier
                    so GradientTape can watch spatial features
                    between the two calls.
    """
    target_layer = model.get_layer(target_layer_name)
    out_tensor   = target_layer.output

    # ── BASELINE: GAP outputs 1-D → split into two models ────
    if len(out_tensor.shape) == 2:
        print(f"  [{model.name}]  GAP detected → building SPLIT model")

        # --- get spatial tensor (output of backbone in outer graph) ---
        spatial_tensor = None
        for node in target_layer._inbound_nodes:
            inp = node.input_tensors
            if not isinstance(inp, list):
                inp = [inp]
            spatial_tensor = inp[0]   # (None, 7, 7, 1280)
            break

        if spatial_tensor is None:
            raise ValueError("Could not find spatial input to GAP")

        # Model A: image → (None, 7, 7, 1280)
        feature_extractor = tf.keras.Model(
            inputs  = model.input,
            outputs = spatial_tensor,
            name    = "feature_extractor"
        )

        
        spatial_inp = tf.keras.Input(shape=spatial_tensor.shape[1:])
        x = spatial_inp
        past_spatial = False
        for layer in model.layers:
            if any(spatial_tensor is t
                   for node in layer._inbound_nodes
                   for t in ([node.output_tensors]
                              if not isinstance(node.output_tensors, list)
                              else node.output_tensors)):
                past_spatial = True  
                continue
            if past_spatial and not isinstance(layer, tf.keras.layers.InputLayer):
                x = layer(x)

        classifier = tf.keras.Model(
            inputs  = spatial_inp,
            outputs = x,
            name    = "classifier"
        )

        print(f"  [{model.name}]  feature_extractor → {spatial_tensor.shape}")
        print(f"  [{model.name}]  classifier        → {classifier.output.shape}")
        return {"type": "split",
                "feature_extractor": feature_extractor,
                "classifier":        classifier}

   
    else:
        grad_model = tf.keras.Model(
            inputs  = model.input,
            outputs = [out_tensor, model.output]
        )
        print(f"  [{model.name}]  joint model  feature_shape={out_tensor.shape}")
        return {"type": "joint", "grad_model": grad_model}


print("Building grad models...")
grad_models = {}
for name, mdl in models_dict.items():
    try:
        grad_models[name] = build_grad_model(mdl, TARGET_LAYERS[name])
    except Exception as e:
        print(f"  [SKIP] {name}: {e}")

print(f"\nReady: {list(grad_models.keys())}")

In [ ]:

def compute_gradcam_plus_plus(model_info, img_input, class_idx=None):
    img_tensor = tf.cast(img_input, tf.float32)

    # ── BASELINE (split) ──────────────────────────────────────
    if model_info["type"] == "split":
        feat_model = model_info["feature_extractor"]
        cls_model  = model_info["classifier"]

        with tf.GradientTape() as tape2:
            with tf.GradientTape() as tape1:
                with tf.GradientTape() as tape0:
                    # Step 1: extract spatial features
                    conv_outputs = tf.cast(feat_model(img_tensor, training=False),
                                           tf.float32)

                    # Step 2: watch BEFORE classifier call
                    tape0.watch(conv_outputs)
                    tape1.watch(conv_outputs)
                    tape2.watch(conv_outputs)

                    # Step 3: run classifier on watched features
                    predictions = tf.cast(cls_model(conv_outputs, training=False),
                                          tf.float32)
                    predictions = tf.reshape(predictions, (1, -1))

                    if class_idx is None:
                        class_idx = int(tf.argmax(predictions[0]))
                    class_score = predictions[0, class_idx]

                grads  = tape0.gradient(class_score, conv_outputs)
            grads2 = tape1.gradient(grads,  conv_outputs)
        grads3 = tape2.gradient(grads2, conv_outputs)

    # ── SE / ECA / CA (joint) ─────────────────────────────────
    else:
        grad_model = model_info["grad_model"]

        with tf.GradientTape() as tape2:
            with tf.GradientTape() as tape1:
                with tf.GradientTape() as tape0:
                    conv_outputs, predictions = grad_model(img_tensor, training=False)
                    conv_outputs = tf.cast(conv_outputs, tf.float32)
                    predictions  = tf.cast(predictions,  tf.float32)
                    predictions  = tf.reshape(predictions, (1, -1))

                    tape0.watch(conv_outputs)
                    tape1.watch(conv_outputs)
                    tape2.watch(conv_outputs)

                    if class_idx is None:
                        class_idx = int(tf.argmax(predictions[0]))
                    class_score = predictions[0, class_idx]

                grads  = tape0.gradient(class_score, conv_outputs)
            grads2 = tape1.gradient(grads,  conv_outputs)
        grads3 = tape2.gradient(grads2, conv_outputs)

    
    if grads is None or grads2 is None or grads3 is None:
        raise ValueError(
            f"Gradients are None. grads={grads is not None}, "
            f"grads2={grads2 is not None}, grads3={grads3 is not None}"
        )

   
    conv_out   = tf.cast(conv_outputs[0], tf.float32)  
    g1 = tf.cast(grads[0],  tf.float32)
    g2 = tf.cast(grads2[0], tf.float32)
    g3 = tf.cast(grads3[0], tf.float32)

    global_sum  = tf.reduce_sum(conv_out, axis=(0, 1))
    alpha_denom = 2.0 * g2 + global_sum * g3 + 1e-7
    alphas      = tf.nn.relu(g2 / alpha_denom)
    weights     = tf.reduce_sum(alphas * tf.nn.relu(g1), axis=(0, 1))

    cam = tf.nn.relu(tf.reduce_sum(weights * conv_out, axis=-1)).numpy()
    cam -= cam.min()
    if cam.max() > 0:
        cam /= cam.max()

    return cam, class_idx

In [ ]:

def overlay_heatmap(cam, img_display, alpha=0.4):
    
    h, w    = img_display.shape[:2]
    cam_r   = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_r), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    return np.clip(alpha * heatmap + (1 - alpha) * img_display, 0, 1)

In [ ]:

def plot_gradcam_all(grad_models, image_files, image_dir, class_names,
                     save_dir='/content/gradcam_outputs'):
    os.makedirs(save_dir, exist_ok=True)
    model_names = list(grad_models.keys())
    n_models    = len(model_names)

    for img_file in image_files:
        img_path               = os.path.join(image_dir, img_file)
        img_input, img_display = preprocess_image(img_path)

        fig, axes = plt.subplots(1, n_models + 1,
                                 figsize=(4.5 * (n_models + 1), 4.5))
        fig.suptitle(f"GradCAM++  |  {img_file}",
                     fontsize=13, fontweight='bold', y=1.01)

       
        axes[0].imshow(img_display)
        axes[0].set_title("Original", fontsize=11, fontweight='bold')
        axes[0].axis('off')

       
        for i, name in enumerate(model_names):
            ax = axes[i + 1]
            try:
                cam, pred_idx = compute_gradcam_plus_plus(grad_models[name], img_input)
                overlay       = overlay_heatmap(cam, img_display)
                pred_label    = class_names.get(pred_idx, str(pred_idx))
                ax.imshow(overlay)
                ax.set_title(f"{name}\nPred: {pred_label}",
                             fontsize=10, fontweight='bold')
            except Exception as e:
                ax.set_title(f"{name}\nERROR", fontsize=10, color='red')
                print(f"[ERROR] {name} / {img_file}: {e}")
            ax.axis('off')

        plt.tight_layout()
        save_path = os.path.join(save_dir,
                                 f"gradcam_{os.path.splitext(img_file)[0]}.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved → {save_path}\n")


plot_gradcam_all(grad_models, image_files, image_dir, CLASS_NAMES)

In [ ]:

def overlay_heatmap(cam, img_display, alpha=0.4):
    h, w    = img_display.shape[:2]
    cam_r   = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_r), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    return np.clip(alpha * heatmap + (1 - alpha) * img_display, 0, 1)

def plot_gradcam_all(grad_models, image_files, image_dir, class_names,
                     save_dir='/content/gradcam_outputs'):
    os.makedirs(save_dir, exist_ok=True)
    model_names = list(grad_models.keys())
    n_models    = len(model_names)

    for img_file in image_files:
        img_path               = os.path.join(image_dir, img_file)
        img_input, img_display = preprocess_image(img_path)

        fig, axes = plt.subplots(1, n_models + 1,
                                 figsize=(4.5 * (n_models + 1), 4.5))
        fig.suptitle(f"GradCAM++  |  {img_file}",
                     fontsize=13, fontweight='bold', y=1.01)

        axes[0].imshow(img_display)
        axes[0].set_title("Original", fontsize=11, fontweight='bold')
        axes[0].axis('off')

        for i, name in enumerate(model_names):
            ax = axes[i + 1]
            try:
                cam, pred_idx = compute_gradcam_plus_plus(grad_models[name], img_input)
                overlay       = overlay_heatmap(cam, img_display)
                pred_label    = class_names.get(pred_idx, str(pred_idx))
                ax.imshow(overlay)
                ax.set_title(f"{name}\nPred: {pred_label}",
                             fontsize=10, fontweight='bold')
            except Exception as e:
                ax.set_title(f"{name}\nERROR", fontsize=10, color='red')
                print(f"[ERROR] {name} / {img_file}: {e}")
            ax.axis('off')

        plt.tight_layout()
        save_path = os.path.join(save_dir,
                                 f"gradcam_{os.path.splitext(img_file)[0]}.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved → {save_path}\n")

plot_gradcam_all(grad_models, image_files, image_dir, CLASS_NAMES)